# From particles to a shell map

This notebook follows the first steps of the pollux pipeline on one shell of a HACC simulation: particles, count map, density contrast, smoothing, tidal field, and comparison to theory. It runs in a few minutes on a login node, at low resolution.

A real run does the same for all shells, then lensing and catalogs, with parsl: see `docs/pipeline.rst`. To explore the outputs of a run, see the next notebooks.

In [ ]:
import os

# be fair to others on a login node, before importing numba and OpenMP libraries
os.environ['OMP_NUM_THREADS'] = os.environ['NUMBA_NUM_THREADS'] = '8'

import matplotlib.pyplot as plt
import numpy as np

import pollux

# outputs of this notebook go to your scratch
root_dir = os.path.expandvars('$PSCRATCH/HACC-HOS-Y1/notebooks/01_shell')

## Read particles

A `Reader` gives access to the snapshots of a simulation. We keep every fifth snapshot; each one is split in sub-volumes.

In [ ]:
reader = pollux.reader.HACCReader(path='/pscratch/sd/d/desc/HOS_Sims/HACC091_1024', run_params='run/indat.params', snap_sub_every_n=5)

print(f"{reader.nsnapshots} snapshots from z = {reader.zsnapshots.min():.2f} to {reader.zsnapshots.max():.2f}, box of {reader.boxsize:.0f} Mpc")

xyz, box_edges = reader.read_sub(10, 0)
print(f"Sub-volume with {len(xyz):,} particles")

thin = xyz[:, 2] - xyz[:, 2].min() < 20.  # slice of 20 Mpc
plt.figure(figsize=(5, 5))
plt.hist2d(xyz[thin, 0], xyz[thin, 1], bins=250, norm='log', cmap='magma')
plt.gca().set(xlabel='x (Mpc)', ylabel='y (Mpc)', aspect='equal');

## Build a shell

A shell is a range of comoving distance around one snapshot. The `ShellBuilder` replicates the periodic box to fill it, with random isometries, and counts particles in Healpix pixels. We use one octant of the sky and a fraction of the particles to go fast.

In [ ]:
shells = pollux.shells.SnapshotShellBuilder(
    reader, nside=256, root_dir=root_dir, seed=0, mask='octant', dtype=np.float32, apply_grid_isometry=True, group_isometries=True, subsample_box=0.1)

ishell = 12
z_edges = shells.shell_edges_redshift
print(f"Shell {ishell}: {z_edges[ishell]:.2f} < z < {z_edges[ishell + 1]:.2f}")

shell = shells.construct_maps(ishell)  # a few minutes: reads all the sub-volumes of the snapshot

## Density contrast and smoothing

`field_hspmap` turns counts into a density contrast. Fields can be smoothed with a Gaussian of fixed comoving scale, here 20 Mpc to see the effect at this resolution. Tidal fields come from the same spherical harmonics.

Below: density contrast, smoothed density contrast, and one tidal field component.

In [ ]:
density = shells.field_hspmap(ishell)
smoothed = shells.compute_maps_from_alms(ishell, smooth_Mpc=20.)
t11, t22, t12 = shells.compute_tidal_tensor_maps(ishell, smooth_Mpc=20.)

shells.plot_maps([density, smoothed, t12], nx=3, save=False, backend='healpy')

## Comparison to theory

Measured angular power spectra compare to predictions from CCL, with the same tracers and binning. Error bars are Gaussian, for the sky fraction of the octant.

In [ ]:
ell, cl = shells.compute_cls(ishell, save=False)
_, cl_theory, dcl = shells.predict_cls(pairs=[(ishell, ishell)], binning='average', error_bars=True)

plt.errorbar(ell, cl[ishell, ishell], dcl[ishell, ishell], fmt='o', label='measured')
plt.plot(ell, cl_theory[ishell, ishell], label='theory')
plt.loglog()
plt.gca().set(xlabel=r'$\ell$', ylabel=r'$C_\ell$')
plt.legend();

## Next steps

The pipeline then sums weighted shells into convergence maps, derives shear maps, and draws galaxy catalogs. The outputs are explored in `02_products.ipynb`, and measurements compared to theory in `03_statistics.ipynb`.